# 04. Accountability Frameworks

## 📚 Learning Objectives

By completing this notebook, you will:
- Apply accountability frameworks to AI systems
- Assign roles, governance, and redress
- Document and audit decision chains

## 🔗 Prerequisites

- ✅ Basic Python
- ✅ Basic NumPy/Pandas (when applicable)

---

---

# 04. Accountability Frameworks

## 🚨 THE PROBLEM: We Need Accountability for AI Decisions

**Remember the limitation from the previous notebook?**

We learned counterfactual analysis for "what if" explanations. But we discovered:

**How do we ensure accountability and responsibility for AI decisions?**

**The Problem**: Transparent AI systems also need:
- ❌ **Accountability frameworks** (who is responsible?)
- ❌ **Responsibility mechanisms** (how to assign responsibility?)
- ❌ **Audit trails** (how to track decisions?)
- ❌ **Stakeholder accountability** (who answers for outcomes?)

**We've learned:**
- ✅ How to use SHAP for explanations (Notebook 1)
- ✅ How to use LIME for fast explanations (Notebook 2)
- ✅ How to use counterfactuals for "what if" scenarios (Notebook 3)
- ✅ Multiple explanation methods

**But we haven't learned:**
- ❌ How to **define stakeholder responsibilities**
- ❌ How to **create audit trails**
- ❌ How to **establish responsibility mechanisms**
- ❌ How to **ensure accountability** for AI decisions

**We need accountability frameworks** to:
1. Define stakeholder responsibilities
2. Create audit trails
3. Establish responsibility mechanisms
4. Enable accountability for AI decisions

**This notebook solves that problem** by teaching you accountability frameworks for AI systems!

---

## 📚 Prerequisites (What You Need First)

**BEFORE starting this notebook**, you should have completed:
- ✅ **Example 1: SHAP Explanations** - Understanding explainability
- ✅ **Example 2: LIME Explanations** - Understanding local explanations
- ✅ **Example 3: Counterfactual Analysis** - Understanding "what if" scenarios
- ✅ **Basic Python knowledge**: Functions, data manipulation

**If you haven't completed these**, you might struggle with:
- Understanding why accountability matters
- Knowing how to structure accountability frameworks
- Understanding stakeholder responsibilities

---

## 🔗 Where This Notebook Fits

**This is the FOURTH example in Unit 4** - it teaches you accountability!

**Why this example FOURTH?**
- **Before** you can ensure accountability, you need explainability (Examples 1-3)
- **Before** you can implement HITL, you need accountability structures
- **Before** you can build transparent systems, you need accountability

**Builds on**: 
- 📓 Example 1: SHAP Explanations (explainability)
- 📓 Example 2: LIME Explanations (local explanations)
- 📓 Example 3: Counterfactual Analysis ("what if" scenarios)

**Leads to**: 
- 📓 Example 5: Human-in-the-Loop (HITL approaches)
- 📓 Example 6: Transparency Tools (transparency frameworks)

**Why this order?**
1. Accountability provides **responsibility structures** (needed for ethical AI)
2. Accountability teaches **stakeholder roles** (critical for governance)
3. Accountability shows **audit mechanisms** (essential for transparency)

---

## The Story: Who Is Responsible?

Imagine you're using an AI system that makes a wrong decision. **Before** accountability frameworks, you wouldn't know who to hold responsible (developers? data scientists? users?). **After** implementing accountability frameworks, you have clear responsibilities, audit trails, and accountability mechanisms!

Same with AI: **Before** we have explanations but no accountability, now we learn accountability frameworks - define responsibilities, create audit trails, establish accountability! **After** accountability frameworks, we have responsible and accountable AI systems!

---

## Why Accountability Frameworks Matter

Accountability frameworks are essential for ethical AI:
- **Responsibility**: Define who is responsible for AI decisions
- **Transparency**: Enable tracking and auditing of decisions
- **Trust**: Build user confidence through accountability
- **Compliance**: Meet regulatory requirements for accountability
- **Ethics**: Ensure responsible AI development and deployment

## Learning Objectives
1. Understand accountability frameworks
2. Learn stakeholder responsibilities
3. Create audit trails
4. Establish responsibility mechanisms
5. Implement model cards and data sheets
6. Build accountability structures

## 📥 Inputs & 📤 Outputs

**Inputs:** What we use in this notebook

- Libraries and concepts as introduced in this notebook; see prerequisites and code comments.

**Outputs:** What you'll see when you run the cells

- Printed results, figures, and summaries as shown when you run the cells.

---

## Part 1: Accountability as Data Structures

Accountability means three concrete things you can build:
1. A **responsibility matrix** (who is Responsible / Accountable / Consulted / Informed
   for each stage - "RACI")
2. An **audit trail** (every significant decision logged with enough context to reconstruct it)
3. A **redress path** (an appeal route with a named owner)

In [1]:
# Why RACI: accountability fails when it is nobody's job - this matrix forces
# every lifecycle stage to name the human who answers for it.

# Step 1: A RACI responsibility matrix for an AI hiring system
import pandas as pd
from datetime import datetime, timezone

# One row per lifecycle stage; the columns assign who does the work (R),
# who owns the outcome (A), who advises (C), and who must be told (I).
raci = pd.DataFrame([
    ['Data collection',      'Data Engineer',   'Head of Data',    'Legal/DPO',      'HR'],
    ['Model training',       'ML Engineer',     'ML Lead',         'Ethics Board',   'HR'],
    ['Fairness evaluation',  'ML Engineer',     'Ethics Board',    'Legal/DPO',      'Executives'],
    ['Deployment decision',  'ML Lead',         'Product Owner',   'Ethics Board',   'All staff'],
    ['Individual decisions', 'The AI system',   'HR Manager',      'ML Lead',        'Applicant'],
    ['Appeals / redress',    'HR Manager',      'Head of HR',      'Legal/DPO',      'Applicant'],
], columns=['Stage', 'Responsible', 'Accountable', 'Consulted', 'Informed'])

print("RACI RESPONSIBILITY MATRIX - AI Hiring System")
print("=" * 95)
print(raci.to_string(index=False))
print("\nKey rule: 'Accountable' is always a PERSON or a named role - never")
print("'the algorithm'. Row 5 makes that explicit: the system is responsible for")
print("producing the decision, but a human (HR Manager) is accountable for it.")

RACI RESPONSIBILITY MATRIX - AI Hiring System
               Stage   Responsible   Accountable    Consulted   Informed
     Data collection Data Engineer  Head of Data    Legal/DPO         HR
      Model training   ML Engineer       ML Lead Ethics Board         HR
 Fairness evaluation   ML Engineer  Ethics Board    Legal/DPO Executives
 Deployment decision       ML Lead Product Owner Ethics Board  All staff
Individual decisions The AI system    HR Manager      ML Lead  Applicant
   Appeals / redress    HR Manager    Head of HR    Legal/DPO  Applicant

Key rule: 'Accountable' is always a PERSON or a named role - never
'the algorithm'. Row 5 makes that explicit: the system is responsible for
producing the decision, but a human (HR Manager) is accountable for it.


In [2]:
# Why audit trails: when a decision is challenged months later, the trail is
# the only way to reconstruct WHO/WHAT/WHEN - no trail, no accountability.

# Step 2: An audit trail - log every significant automated decision
import hashlib, json

audit_log = []

def log_decision(model_version, applicant_features, prediction, confidence,
                 explanation, human_reviewer=None):
    """Append one reconstructable decision record to the audit trail."""
    record = {
        'timestamp': datetime.now(timezone.utc).isoformat(timespec='seconds'),
        'model_version': model_version,
        'input_hash': hashlib.sha256(
            json.dumps(applicant_features, sort_keys=True).encode()
        ).hexdigest()[:12],
        'prediction': prediction,
        'confidence': confidence,
        'top_factors': explanation,
        'human_reviewer': human_reviewer,   # None = fully automated
    }
    audit_log.append(record)
    return record

# Simulate three decisions passing through the system
log_decision('hiring-model-v2.3', {'income': 62, 'years': 8},  'shortlist', 0.91,
             ['years_employed +', 'income +'])
log_decision('hiring-model-v2.3', {'income': 31, 'years': 1},  'reject',    0.55,
             ['income -', 'years_employed -'], human_reviewer='hr.manager@corp')
log_decision('hiring-model-v2.3', {'income': 48, 'years': 4},  'shortlist', 0.78,
             ['income +'])

print("AUDIT TRAIL")
print("=" * 95)
for r in audit_log:
    print(f"  {r['timestamp']} | {r['model_version']} | input {r['input_hash']} | "
          f"{r['prediction']:<9} | conf {r['confidence']:.2f} | "
          f"reviewer: {r['human_reviewer'] or 'AUTOMATED'}")

# The auditor's query: automated decisions the model was unsure about -
# exactly the cases that policy says should have had human review.
low_conf_auto = [r for r in audit_log
                 if r['confidence'] < 0.7 and r['human_reviewer'] is None]
print(f"\nAudit query: low-confidence decisions WITHOUT human review: "
      f"{len(low_conf_auto)}")
print("This is exactly the query an auditor runs - and why the trail must exist.")
print("(Notice decision 2: low confidence, so a human reviewer signed off.)")

AUDIT TRAIL
  2026-08-23T16:31:09+00:00 | hiring-model-v2.3 | input 050a6392a590 | shortlist | conf 0.91 | reviewer: AUTOMATED
  2026-08-23T16:31:09+00:00 | hiring-model-v2.3 | input d511a355812c | reject    | conf 0.55 | reviewer: hr.manager@corp
  2026-08-23T16:31:09+00:00 | hiring-model-v2.3 | input bba40b5896a3 | shortlist | conf 0.78 | reviewer: AUTOMATED

Audit query: low-confidence decisions WITHOUT human review: 0
This is exactly the query an auditor runs - and why the trail must exist.
(Notice decision 2: low confidence, so a human reviewer signed off.)


---

## 🚫 When Accountability Frameworks Hit a Limitation

### The Limitation We Discovered

We've learned accountability frameworks for defining responsibilities. **But there's still a challenge:**

**How do we incorporate human judgment into AI decision-making?**

Accountability frameworks work well when:
- ✅ We have clear responsibilities defined
- ✅ We have audit trails in place
- ✅ We have accountability mechanisms

**But ethical AI systems also need:**
- ❌ **Human oversight** (human judgment for critical decisions)
- ❌ **Human-in-the-loop** (HITL) approaches
- ❌ **Human review** for uncertain cases
- ❌ **Human validation** of AI decisions

### Why This Is a Problem

When we have accountability but no human oversight:
- Critical decisions may be made without human judgment
- Uncertain cases may not get human review
- AI decisions may lack human validation
- We may miss important context that humans understand

### The Solution: Human-in-the-Loop (HITL) Approaches

We need **human-in-the-loop approaches** to:
1. Incorporate human judgment into AI decisions
2. Enable human review for uncertain cases
3. Provide human oversight for critical decisions
4. Combine AI efficiency with human judgment

**This is exactly what we'll learn in the next notebook: Human-in-the-Loop Approaches!**

---

## ➡️ Next Steps

**You've completed this notebook!** Now you understand:
- ✅ How to use SHAP, LIME, and counterfactuals (Notebooks 1-3)
- ✅ How to establish accountability frameworks (This notebook!)
- ✅ **The limitation**: We need human oversight!

**Next notebook**: `05_hitl_approaches.ipynb`
- Learn about human-in-the-loop approaches
- Understand human oversight mechanisms
- Implement HITL for critical decisions
- Combine AI with human judgment

## 📚 References

1. Mitchell, M., Wu, S., Zaldivar, A., et al. (2019). *Model Cards for Model Reporting*. FAT* 2019. <https://arxiv.org/abs/1810.03993>
2. Gebru, T., Morgenstern, J., Vecchione, B., et al. (2021). *Datasheets for Datasets*. Communications of the ACM, 64(12). <https://arxiv.org/abs/1803.09010>
3. Raji, I. D., Smart, A., White, R. N., et al. (2020). *Closing the AI Accountability Gap: Defining an End-to-End Framework for Internal Algorithmic Auditing*. FAT* 2020. <https://arxiv.org/abs/2001.00973>